# Human-in-the-loop
This notebook is my exploration of the [Add human-in-the-loop controls](https://langchain-ai.github.io/langgraph/tutorials/get-started/4-human-in-the-loop/) tutorial by Langgraph. It follows the previous notebook that [added memory](./chatbot_with_memory_langgraph.ipynb) to a chatbot.

## Required packages
* `langchain[anthropic]`
* `langchain-tavily`
* `langgraph`
* `langgraph-checkpoint-sqlite`
* `langsmith`

In [2]:
import json
import sqlite3
import uuid
from pprint import pprint
from typing import Annotated
from typing_extensions import TypedDict

from IPython.display import display, Markdown
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver 
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

In [3]:
# Load Anthropic and Tavily API keys
# The `secrets.env` file is expected to have the following lines:
# ANTHROPIC_API_KEY="sk-ant-xxxxx"
# TAVILY_API_KEY="tvly-dev-xxxxx"
from dotenv import load_dotenv
load_dotenv("../../secrets.env")

True

## Build a chatbot with tool access and memory
Like we did previously, let us create a chatbot graph that can use the web search tool as needed and persist the conversation in a SQLite database.

In [4]:
search_tool = TavilySearch(max_results=2)

## Human assistance tool
In Langgraph, human input during graph execution can be provided as a tool. Given the right prompt (like specifying the LLM to seek human feedback in case of any uncertainty), the LLM will call the human assistance tool. In this tool, we call the `interrupt()` function that interrupts the execution of the graph. But before the interruption, the LLM generates a query that it wants the human to answer which is returned as part of the `data` attribute of the response.

In [5]:
@tool
def human_assistance(query: str) -> str:
    """A tool that allows the user to provide assistance."""
    display(Markdown(f"### Human Assistance Needed\n{query}"))
    response = interrupt({"query": query})
    return response["data"]

## Build the graph
We now attach both tools to the LLM, define the chatbot function, and build the graph.

In [6]:
tools = [search_tool, human_assistance]

In [7]:
llm = init_chat_model("anthropic:claude-sonnet-4-0")

class State(TypedDict):
    messages: Annotated[list, add_messages]

# Tell the LLM the tools it can call
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    # Since the human assistance tool might be called, we avoid parallel
    # tool calls.
    assert len(message.tool_calls) <= 1, "Only one tool call is allowed at a time"
    return {"messages": [message]}

In [8]:
graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition
)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

Before compiling the graph, we provide it memory in the form of a SQLite database.

In [9]:
# Note: check_same_thread=False is OK as the implementation uses a lock
# to ensure thread safety.
conn = sqlite3.connect("chatbot_human_in_loop.db", check_same_thread=False)
memory = SqliteSaver(conn)

graph = graph_builder.compile(checkpointer=memory)

## Prompt the chatbot
We now define a unique `thread_id` for each conversation in order to return to the previous state of the conversation. We then pass the initial query which is *"What was the last game played by Arsenal FC? Ask for assistance if you are not sure what date it is today."*. This explicitly informs the LLM to call the human assistance tool if it needs to know the current date. Since the code below cannot handle the input to this query from the LLM, it throws a `TypeError` which we capture and display that human assistance is needed.

In [32]:
config = {"configurable": {"thread_id": uuid.uuid4()}}

In [33]:
def display_text_response(text: str) -> None:
    """Display a Markdown-formatted assistant response."""
    display(Markdown("**Assistant:**\n" + text))

def display_tool_use(tool_info: dict) -> None:
    """Display information about a tool usage in Markdown format.
    
    Args:
        tool_info (dict): Dictionary containing tool usage details.
    """
    tool_name = tool_info.get("name", "")
    tool_input = tool_info.get("input", {})
    query = tool_input.get("query", "")
    topic = tool_input.get("topic", "")
    search_depth = tool_input.get("search_depth", "")
    display(Markdown(
        f"**Tool Use:** `{tool_name}`\n\n"
        f"- **Query:** `{query}`\n"
        f"- **Topic:** `{topic}`\n"
        f"- **Search Depth:** `{search_depth}`\n"
    ))

def display_search_results(results: list) -> None:
    """Display a list of search results in Markdown format.
    
    Args:
        results (list): List of dictionaries containing search result details.
    """
    for result in results:
        url = result.get("url", "")
        title = result.get("title", "")
        score = result.get("score", "")
        published_date = result.get("published_date", "")
        display(Markdown(
            f"**Search Result:**\n"
            f"- **Title:** [{title}]({url})\n"
            f"- **Score:** `{score}`\n"
            f"- **Published Date:** `{published_date}`\n"
        ))

def handle_latest_message(latest_message: object) -> None:
    """Handle and display the latest message from the graph event.
    
    Args:
        latest_message (object): The latest message, can be a list or string.
    """
    if isinstance(latest_message, list):
        for v in latest_message:
            if v["type"] == "text":
                display_text_response(v["text"])
            elif v["type"] == "tool_use":
                display_tool_use(v)
            else:
                raise ValueError("Message type is neither 'text' nor 'tool_use'")
    elif isinstance(latest_message, str):
        try:
            results = json.loads(latest_message)["results"]
            display_search_results(results)
        except json.JSONDecodeError:
            display_text_response(latest_message)
    else:
        raise ValueError("Latest message is neither of 'str' or 'list'")

def stream_graph_updates(user_input: Union[str, Command], config: dict) -> None:
    """Stream updates from the graph for a given user input and display responses.
    
    Args:
        user_input (Union[str, Command]): The user's input message or command.
    """
    if isinstance(user_input, str):
        stream_input = {"messages": [{"role": "user", "content": user_input}]}
    elif isinstance(user_input, Command):
        stream_input = user_input
    else:
        raise ValueError("Invalid user input type")

    for event in graph.stream(
        stream_input,
        config=config
    ):
        for value in event.values():
            latest_message = value["messages"][-1].content
            handle_latest_message(latest_message)

while True:
    user_input = input("User: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    display(Markdown(f"**Question**:\n{user_input}\n"))
    try:
        stream_graph_updates(user_input, config)
    except TypeError:
        display(Markdown(f"Waiting for human assistance..."))
        break


**Question**:
What was the last game played by Arsenal FC? Ask for assistance if you are not sure what date it is today.


**Assistant:**
I need to know what today's date is to search for Arsenal FC's most recent game effectively. Let me ask for assistance with this.

**Tool Use:** `human_assistance`

- **Query:** `What is today's date? I need this to search for Arsenal FC's most recent game.`
- **Topic:** ``
- **Search Depth:** ``


### Human Assistance Needed
What is today's date? I need this to search for Arsenal FC's most recent game.

Waiting for human assistance...

## Resume execution
Once the LLM seeks human assistance and interrupts the execution of the graph, we can resume it by passing a `Command` object. In this object, we pass the `resume` argument to be a dictionary with the expected `data` key whose value is the answer to the query posed by the LLM.

Previously, we updated the `stream_graph_updates()` function to handle string and `Command` object inputs separately. Hence, we pass it the `Command` object as is along with the configuration dictionary which contains the thread ID. 

Once provided with the current date, the LLM proceeds to using the Tavily search tool to get the latest results of Arsenal FC and give the final answer.

In [34]:
date_response = Command(resume={"data": "Today is August 23, 2025."})
stream_graph_updates(date_response, config)

### Human Assistance Needed
What is today's date? I need this to search for Arsenal FC's most recent game.

**Assistant:**
Today is August 23, 2025.

**Assistant:**
Now I'll search for Arsenal FC's most recent game:

**Tool Use:** `tavily_search`

- **Query:** `Arsenal FC last game recent match result 2025`
- **Topic:** `news`
- **Search Depth:** `basic`


**Search Result:**
- **Title:** [Arsenal FC vs. Leeds United Prediction, Odds, Picks - Aug 23 - FOX Sports](https://www.foxsports.com/articles/soccer/arsenal-fc-vs-leeds-united-prediction-odds-picks-aug-23)
- **Score:** `0.7491794`
- **Published Date:** `Thu, 21 Aug 2025 18:02:36 GMT`


**Search Result:**
- **Title:** [Gyokeres scores his first goals for Arsenal in 5-0 rout of Leeds, impressive Tottenham beats City - AP News](https://apnews.com/article/premier-league-man-city-tottenham-arsenal-leeds-b3136d24aebe59fb119934b4bae28f6c)
- **Score:** `0.71496695`
- **Published Date:** `Sat, 23 Aug 2025 19:50:00 GMT`


**Assistant:**
Based on the search results, Arsenal FC's most recent game was today (August 23, 2025) against Leeds United. Arsenal won the match 5-0 at the Emirates Stadium in London. 

Key details from the match:
- **Final Score**: Arsenal 5-0 Leeds United
- **Date**: August 23, 2025 (today)
- **Venue**: Emirates Stadium, London
- **Notable performance**: Viktor Gyökeres scored multiple goals for Arsenal, including what appears to be his first goals for the club, with at least one coming from a penalty spot

This was a dominant performance by Arsenal in their Premier League match against Leeds United.